In [1]:
import pandas as pd
import warnings

from notebooks.consts import ORIGINAL_OLIGO_CSV, PROCESSED_OLIGO_CSV_GZ
from tauso.off_target.search import find_all_gene_off_targets

# Configuration
AMBIGUOUS_CSV = "ambiguous_target_gene_mappings.csv"

CONTEXT_COL = "rna_context"
TARGET_GENE_COL = "target_gene"

GENOME = "GRCh38"
MAX_MISMATCHES = 0  # for >200bp context, exact match only

In [2]:
df = pd.read_csv(ORIGINAL_OLIGO_CSV)

missing_cols = {CONTEXT_COL, TARGET_GENE_COL} - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df.head()

,aso_sequence_5_to_3,inhibition_percent,chemistry,custom_id,target_mrna,target_gene,cell_line,cell_line_species,dosage,cells_per_well,transfection_method,steric_blocking,rna_context,sugar_mods,backbone_mods
0,GCAGATATTTCAATATACAG,84.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2023...,APP,APP,SH-SY5Y,human,4000.0,20000.0,Electroporation,False,UCCAUAUCUAUUGGUAUAAUAAAGCUGAUAUUUAGAGAGAACAUUA...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PO', 'PO', 'PO', 'PS', 'PS', 'PS', 'PS..."
1,GCTCATTATCTCATTTGACT,46.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2023...,APP,APP,SH-SY5Y,human,4000.0,20000.0,Electroporation,False,CAUGUGCUUUGGAAUAACACUCUUGAGUAGACUCUAGGCGGGUUUU...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PO', 'PO', 'PO', 'PS', 'PS', 'PS', 'PS..."
2,GCTGACAAACTGTACTGGGT,69.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2023...,APP,APP,SH-SY5Y,human,4000.0,20000.0,Electroporation,False,UGGAUUAUUUAGUUAAUGGUGUUGGGUUAUCUGACUGUACAUAGAA...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PO', 'PO', 'PO', 'PS', 'PS', 'PS', 'PS..."
3,CCACTCAATATCCTACCTCT,44.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2023...,APP,APP,SH-SY5Y,human,4000.0,20000.0,Electroporation,False,UAGUGACACCUCGAUGCAAGGGAGCAUGUAAAUGUGCUCUAUUAGC...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PO', 'PO', 'PO', 'PS', 'PS', 'PS', 'PS..."
4,CTACAGGTTTTCCCCACATC,53.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2023...,APP,APP,SH-SY5Y,human,4000.0,20000.0,Electroporation,False,CAUUUUAAAAUUUAUUUUAAUUCCUUACCUAUUUUCUUGUUCAAUU...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PO', 'PO', 'PO', 'PS', 'PS', 'PS', 'PS..."


In [3]:
# Extract one context sequence per target gene to minimize search space
gene_to_context = (
    df.dropna(subset=[TARGET_GENE_COL, CONTEXT_COL])
    .groupby(TARGET_GENE_COL)[CONTEXT_COL]
    .first()
)

print(f"Resolving {len(gene_to_context)} unique target genes")

Resolving 293 unique target genes


In [4]:
gene_to_canonical = {}
ambiguous_genes = {}

for i, (target_gene, rna_seq) in enumerate(gene_to_context.items(), start=1):
    dna = rna_seq.upper().replace("U", "T")

    hits = find_all_gene_off_targets(dna, genome=GENOME, max_mismatches=MAX_MISMATCHES)

    genes = hits["gene_name"].dropna().unique().tolist() if not hits.empty else []

    if len(genes) == 0:
        gene_to_canonical[target_gene] = pd.NA

    elif len(genes) == 1:
        gene_to_canonical[target_gene] = genes[0]

    else:
        gene_to_canonical[target_gene] = ";".join(genes)
        ambiguous_genes[target_gene] = genes

    if i % 50 == 0 or i == len(gene_to_context):
        print(f"[{i}/{len(gene_to_context)}] genes processed")

[50/293] genes processed
[100/293] genes processed
[150/293] genes processed
[200/293] genes processed
[250/293] genes processed
[293/293] genes processed


In [5]:
from tauso.data.consts import CANONICAL_GENE

# Map canonical genes back to the main dataframe
df[CANONICAL_GENE] = df[TARGET_GENE_COL].map(gene_to_canonical)

# Save main output
df.to_csv(PROCESSED_OLIGO_CSV_GZ, index=False)
print(f"Saved annotated dataframe → {PROCESSED_OLIGO_CSV_GZ}")

# Save and warn about ambiguous mappings
if ambiguous_genes:
    warnings.warn(
        f"{len(ambiguous_genes)} target genes mapped to multiple canonical genes"
    )
else:
    print("No ambiguous gene mappings found 🎉")

amb_df = pd.DataFrame(
    [
        {TARGET_GENE_COL: tg, "matched_canonical_genes": ";".join(genes)}
        for tg, genes in ambiguous_genes.items()
    ]
)


Saved annotated dataframe → /home/michael/career/tauso_article/tauso_source2/notebooks/data/OligoAI/raw_data/aso_inhibitions_with_canonical_gene.csv.gz


/tmp/ipykernel_458852/4275408560.py:12: UserWarning: 3 target genes mapped to multiple canonical genes
  warnings.warn(


In [6]:
amb_df

,target_gene,matched_canonical_genes
0,DUX4,DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
1,NAIP,NAIPP3;NAIPP1;NAIPP2;NAIPP4;NAIP
2,NOTCH2,NOTCH2NLC;NOTCH2;NBPF10;NBPF14


In [7]:
# Now we check what happened with the errors

In [8]:
not_found = df[df[CANONICAL_GENE].isna()]

print(f"{len(not_found)} RNA contexts were not found in the genome")

0 RNA contexts were not found in the genome


In [9]:
multi_gene_mask = (
    df["Canonical Gene Name"]
    .notna()
    & df["Canonical Gene Name"].str.contains(";", regex=False)
)

count_multi = multi_gene_mask.sum()

print(f"{count_multi} rows have more than one canonical gene")

785 rows have more than one canonical gene


In [10]:
df[multi_gene_mask]

,aso_sequence_5_to_3,inhibition_percent,chemistry,custom_id,target_mrna,target_gene,cell_line,cell_line_species,dosage,cells_per_well,transfection_method,steric_blocking,rna_context,sugar_mods,backbone_mods,Canonical Gene Name
15286,TCGTCCCCGGGCTTCCGCGG,31.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,AUGGCCCUCCCGACACCCUCGGACAGCACCCUCCCCGCGGAAGCCC...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15287,AAACGAGTCTCCGTCGCCGT,48.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,AUGGCCCUCCCGACACCCUCGGACAGCACCCUCCCCGCGGAAGCCC...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15288,AGCAGGCTCGCAGGGCCTCG,0.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,AUGGCCCUCCCGACACCCUCGGACAGCACCCUCCCCGCGGAAGCCC...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15289,GTGGCGATGCCCGGGTACGG,33.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,CAGCACCCUCCCCGCGGAAGCCCGGGGACGAGGACGGCGACGGAGA...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15290,GGCCTGGGCCAGCCGTTCTC,52.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,CGGGGACGAGGACGGCGACGGAGACUCGUUUGGACCCCGAGCCAAA...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137094,TGAATCCTGGACTCCG,92.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,GUCACCGGAUCCCAGACCGCCCUGCUCCUCCGAGCCUUUGAGAAGG...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
137095,CAGATCTGAATCCTGG,82.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,GGAUCCCAGACCGCCCUGCUCCUCCGAGCCUUUGAGAAGGAUCGCU...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
137096,CCAGATCTGAATCCTG,90.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,GAUCCCAGACCGCCCUGCUCCUCCGAGCCUUUGAGAAGGAUCGCUU...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
137097,TCGATTCTGAAACCAG,93.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,CCCUGCUCCUCCGAGCCUUUGAGAAGGAUCGCUUUCCAGGCAUCGC...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...


In [11]:
# Show all inconsistencies between the target gene and the canonical gene name
df[df[CANONICAL_GENE] != df['target_gene']][[CANONICAL_GENE, 'target_gene']].drop_duplicates()

,Canonical Gene Name,target_gene
15286,DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...,DUX4
24132,RP11-739B23.1,PAK1
47040,NOTCH2NLC;NOTCH2;NBPF10;NBPF14,NOTCH2
67898,DNMT1,S1PR2
77552,RP11-823E8.3,DUSP6
124513,MIR6801,PPP2R1A
129677,ARL14EPP1,EIF2AK2
141396,NAIPP3;NAIPP1;NAIPP2;NAIPP4;NAIP,NAIP
150782,IGF2-AS,IGF2


In [12]:
df[df[CANONICAL_GENE].str.contains(';')]

,aso_sequence_5_to_3,inhibition_percent,chemistry,custom_id,target_mrna,target_gene,cell_line,cell_line_species,dosage,cells_per_well,transfection_method,steric_blocking,rna_context,sugar_mods,backbone_mods,Canonical Gene Name
15286,TCGTCCCCGGGCTTCCGCGG,31.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,AUGGCCCUCCCGACACCCUCGGACAGCACCCUCCCCGCGGAAGCCC...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15287,AAACGAGTCTCCGTCGCCGT,48.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,AUGGCCCUCCCGACACCCUCGGACAGCACCCUCCCCGCGGAAGCCC...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15288,AGCAGGCTCGCAGGGCCTCG,0.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,AUGGCCCUCCCGACACCCUCGGACAGCACCCUCCCCGCGGAAGCCC...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15289,GTGGCGATGCCCGGGTACGG,33.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,CAGCACCCUCCCCGCGGAAGCCCGGGGACGAGGACGGCGACGGAGA...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
15290,GGCCTGGGCCAGCCGTTCTC,52.0,length=20 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,100.0,6000.0,Lipofection,False,CGGGGACGAGGACGGCGACGGAGACUCGUUUGGACCCCGAGCCAAA...,"['MOE', 'MOE', 'MOE', 'MOE', 'MOE', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
137094,TGAATCCTGGACTCCG,92.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,GUCACCGGAUCCCAGACCGCCCUGCUCCUCCGAGCCUUUGAGAAGG...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
137095,CAGATCTGAATCCTGG,82.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,GGAUCCCAGACCGCCCUGCUCCUCCGAGCCUUUGAGAAGGAUCGCU...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
137096,CCAGATCTGAATCCTG,90.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,GAUCCCAGACCGCCCUGCUCCUCCGAGCCUUUGAGAAGGAUCGCUU...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
137097,TCGATTCTGAAACCAG,93.0,length=16 modifications=[Modification(modifica...,patent-scrape-v2/data/inhibition_tables/US2024...,DUX4,DUX4,54-2,human,200.0,10000.0,Lipofection,False,CCCUGCUCCUCCGAGCCUUUGAGAAGGAUCGCUUUCCAGGCAUCGC...,"['CET', 'CET', 'CET', 'DNA', 'DNA', 'DNA', 'DN...","['PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS', 'PS...",DUX4L8;DUX4;DUX4L2;DUX4L3;DUX4L6;DUX4L1;DUX4L5...
